# QUERY ENGINE = Question → Answer
# CHAT ENGINE = Conversation → Context-aware Answer

In [ ]:
#!pip install -U llama-index
#!pip install -U llama-index-llms-openai
#!pip install -U llama-index-embeddings-openai
#!pip install -U llama-index-readers-file
#!pip install -U pypdf python-dotenv

# IMPORTS

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from llama_index.core import (
    Settings,
    SimpleDirectoryReader,
    StorageContext,
    VectorStoreIndex,
    load_index_from_storage
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.vector_stores import (
    MetadataFilter,
    MetadataFilters,
    FilterOperator
)
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.chat_engine import ContextChatEngine
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

# CONFIGURATION

In [2]:
load_dotenv()
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)    # Settings contains default models and configuration that other LlamaIndex components can use.
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

C:\Users\User\anaconda3\envs\ai\Lib\site-packages\pydantic\_internal\_generate_schema.py:937: UserWarning: Mixing V1 models and V2 models (or constructs, like `TypeAdapter`) is not supported. Please upgrade `BasePromptTemplate` to V2.
  warnings.warn(


# LOAD DATA

In [3]:
pdf_path = next(Path("./data").glob("*.pdf"))
documents = SimpleDirectoryReader(input_files=["./data/HopeAI.pdf"]).load_data()
print("Pages loaded:", len(documents))

2026-09-24 17:09:55,618 - INFO - NumExpr defaulting to 12 threads.


Pages loaded: 4


# CHUNKING

In [4]:
text_splitter = SentenceSplitter(
    chunk_size=500,
    chunk_overlap=50
)
Settings.text_splitter = text_splitter

# EMBEDDINGS

In [5]:
embedding_model = OpenAIEmbedding(model="text-embedding-3-small")

# ADD METADATA

In [6]:
# Add custom metadata to every PDF page
for document in documents:
    document.metadata.update(
        {
            "source_file": pdf_path.name,
            "document_type": "PDF",
            "collection": "HopeAI"
        }
    )

# Split PDF pages into smaller chunks
nodes = text_splitter.get_nodes_from_documents(
    documents
)

print("PDF loaded:", pdf_path.name)
print("PDF pages loaded:", len(documents))
print("Chunks created:", len(nodes))

# Display sample metadata
if nodes:
    print("Sample metadata:", nodes[0].metadata)

PDF loaded: HopeAI.pdf
PDF pages loaded: 4
Chunks created: 4
Sample metadata: {'page_label': '1', 'file_name': 'HopeAI.pdf', 'file_path': 'data\\HopeAI.pdf', 'file_type': 'application/pdf', 'file_size': 232790, 'creation_date': '2026-09-24', 'last_modified_date': '2026-09-10', 'source_file': 'HopeAI.pdf', 'document_type': 'PDF', 'collection': 'HopeAI'}


# BUILDING A VECTOR INDEX

In [7]:
index = VectorStoreIndex(
    nodes=nodes,
    embed_model=embedding_model,
    show_progress=True
)

print("Vector index created.")

Generating embeddings:   0%|          | 0/4 [00:00<?, ?it/s]

2026-09-24 17:10:11,770 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Vector index created.


# CREATE METADATA FILTERS

In [8]:
metadata_filters = MetadataFilters(
    filters=[
        MetadataFilter(
            key="source_file",
            value="HopeAI.pdf",
            operator=FilterOperator.EQ
        ),
        MetadataFilter(
            key="document_type",
            value="PDF",
            operator=FilterOperator.EQ
        )
    ]
)

print("Metadata filters created:")
print("- source_file = HopeAI.pdf")
print("- document_type = PDF")

Metadata filters created:
- source_file = HopeAI.pdf
- document_type = PDF


# RETRIEVAL

In [9]:
retriever = index.as_retriever(
    similarity_top_k=3,
    filters=metadata_filters
)

print("Retriever created.")
print("Number of chunks to retrieve:", 3)

Retriever created.
Number of chunks to retrieve: 3


# CEATE QUERY ENGINE

In [10]:
query_engine = RetrieverQueryEngine.from_args(
    retriever=retriever,
    llm=Settings.llm,
    response_mode="compact"
)

print("Query engine created.")

Query engine created.


# ASK QUERY ENGINE

In [11]:
question = input("Enter your question: ")

response = query_engine.query(question)

print("\nAnswer:")
print(response)

Enter your question:  Who is the tutor of Hope AI?


2026-09-24 17:10:35,169 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-09-24 17:10:36,614 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



Answer:
The tutor of Hope AI is Ramisha Rani K.


# CREATE CHAT ENGINE

In [12]:
chat_engine = ContextChatEngine.from_defaults(
    retriever=retriever,
    llm=Settings.llm,
    system_prompt=(
        "You are a helpful assistant answering questions about "
        "HopeAI.pdf. Answer using only the retrieved PDF context. "
        "If the answer is unavailable, say: "
        "'The answer was not found in HopeAI.pdf.'"
    )
)

print("Chat engine created.")

Chat engine created.


# START THE CONVERSATIONAL CHATBOT

In [13]:
print("HopeAI PDF Chatbot")
print("Type 'quit' to stop.")
print("Type 'reset' to clear the conversation history.")

while True:
    user_question = input("\nYou: ").strip()

    if not user_question:
        continue

    if user_question.lower() == "quit":
        print("Chat ended.")
        break

    if user_question.lower() == "reset":
        chat_engine.reset()
        print("Conversation history cleared.")
        continue

    response = chat_engine.chat(user_question)

    print("\nAssistant:")
    print(response)

HopeAI PDF Chatbot
Type 'quit' to stop.
Type 'reset' to clear the conversation history.



You:  Who is the tutor of Hope AI?


2026-09-24 17:10:46,988 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-09-24 17:10:49,042 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



Assistant:
The tutor of Hope AI is Ramisha Rani K, who has over 8 years of experience in AI and Data Science.



You:  What is the name of the company?


2026-09-24 17:10:51,667 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-09-24 17:10:52,729 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



Assistant:
The name of the company is RamishaHope Artificial Intelligence Private Limited.



You:  quit


Chat ended.


In [ ]:
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.response_synthesizers import get_response_synthesizer

query_engine = index.as_query_engine(
    similarity_top_k=3,
    response_mode="compact"
)

query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer
)

response = query_engine.query(
    "Who is the tutor of Hope AI?."
)

print(response)